<a href="https://colab.research.google.com/github/zillioxmatthewprojects-ops/Bus_Route_ML/blob/main/Bus_Route_Data_Creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============SETUP=========================================
!pip install openpyxl pandas numpy --quiet

import pandas as pd
import numpy as np

# ==============CONTROLS=========================================
N_ROWS = 25000 #25k if ML 5k for data cleaning
rng = np.random.default_rng(28) #Seed number
CLEAN_MODE = False # Master override - flips every toggle off for the clean ML export
RUN_LABEL = "Bus Depot ML2" if CLEAN_MODE else "Bus Depot"

# ==============THE WORLD=========================================
# Swap this whole block to change what the dataset is about (cars, orders,
# whatever) - nothing below this needs to know or care.
# Keep a mix of multi-word ("New York") and single-word ("Texas") entity
# names - the naming-inconsistency error needs multi-word names to matter.
WORLD = {
    "entity_col": "State",
    "route_col": "RouteName",
    "entities": ["California", "Texas", "New York", "Florida", "Maryland", "Colorado"],
    "routes_by_entity": {
        "California": ["Route 1", "Route 2"],
        "Texas": ["Route 3", "Route 4"],
        "New York": ["Route 5", "Route 6"],
        "Florida": ["Route 7"],
        "Maryland": ["Route 8"],
        "Colorado": ["Route 9"],
    },
    "vehicle_types": ["Standard", "HellCat", "Mini", "Volt"], # Bus categories
    "vehicle_multiplier": {"Mini": 0.5, "Standard": 1.0, "HellCat": 1.5, "Volt": 1.2},
    "vehicle_capacity": {"Mini": 35, "Standard": 70, "HellCat": 105, "Volt": 85},
    "vehicle_weights_by_tier": {#Standard, HellCat, Mini, Volt  — matches vehicle_types order
    # Vehicle mix depends on route demand: busy routes lean toward big buses,
    # quiet routes toward Mini. Creates a learnable route<->vehicle relationship
    # (and the oversized-bus inefficiencies the analysis is meant to find).
    "low":  [0.25, 0.15, 0.50, 0.10],
    "mid":  [0.35, 0.25, 0.30, 0.10],
    "high": [0.40, 0.35, 0.10, 0.15]},
    # Route-level demand baseline. Higher = busier corridor.
    # Kept within 0.85-1.15: wider spreads stack with vehicle_multiplier and
    # seasonality and push ridership into the capacity ceiling too often.
    "route_multiplier": {
        "Route 1": 1.08, "Route 2": 1.02,   # California
        "Route 3": 0.92, "Route 4": 0.88,   # Texas
        "Route 5": 1.15, "Route 6": 1.10,   # New York
        "Route 7": 0.85,                    # Florida
        "Route 8": 1.00,                    # Maryland - baseline
        "Route 9": 0.95,                    # Colorado
    },
    # Route picked directly by weight instead of via state, so volume is a
    # deliberate choice, not a side effect of how many routes each state has.
    # Dual-route states split unevenly (in-state competition); Florida bumped
    # above the single-route baseline; Maryland/Colorado are the floor.
    "route_weights": {
        "Route 1": 0.16, "Route 2": 0.09,   # California
        "Route 3": 0.14, "Route 4": 0.07,   # Texas
        "Route 5": 0.12, "Route 6": 0.09,   # New York
        "Route 7": 0.15,                    # Florida
        "Route 8": 0.08,                    # Maryland
        "Route 9": 0.10,                    # Colorado
    },
    # Q1 cold + resolutions, Q2 people walk/bike, Q3 heat vs travel evens out,
    # Q4 cold returns plus holiday travel. Averages to 0.99 so annual volume
    # stays near flat while the shape moves.
    "seasonality_by_quarter": {1: 1.06, 2: 0.90, 3: 0.98, 4: 1.03},
    # only entities listed here get the naming-inconsistency error
    "entity_naming_variants": {
        "California": ["california", "CA", "Calif."],
        "Texas": ["texas", "TX"],
        "New York": ["new york", "NY"],
        "Florida": ["florida", "FL"],
    },
}

def demand_tier(route_multiplier):
    # thresholds chosen so your 0.85-1.15 multiplier range splits into
    # 3 groups: <0.95 quiet, 0.95-1.05 average, >1.05 busy
    if route_multiplier < 0.95:
        return "low"
    elif route_multiplier <= 1.05:
        return "mid"
    else:
        return "high"

# ==============ERROR TOGGLES=========================================
# include: on/off. n: how many instances. Each error type is self-contained -
# toggling one off doesn't affect any other.
ERROR_CONFIG = {
    "exact_dupes":          {"include": True,  "n": 8},
    "conflicting_dupes":    {"include": False, "n": 4},
    "state_naming":         {"include": True,  "n": 100},
    "route_naming":         {"include": True,  "n": 100},
    "text_numbers":         {"include": True,  "n": 8},
    "blanks":               {"include": True,  "n": 8},
    "outlier_ridership":    {"include": True,  "n": 4},
    "negative_ridership":   {"include": True,  "n": 10},
    "over_capacity":        {"include": True,  "n": 5},
    "wrong_year_dates":     {"include": False, "n": 3}, #only works with 1 year
    "route_state_mismatch": {"include": True, "n": 5},
    "data_dropout":         {"include": False, "entity": "Maryland",
        # STate and day-of-year the dropout starts. Range 150-250: enough history
        # before it, enough empty space after it to read as a real dropout.
        "cutoff_day_of_year": 180,
    },
}

# Master override - flips every toggle off for the clean ML export
if CLEAN_MODE:
    for cfg in ERROR_CONFIG.values():
        cfg["include"] = False

# ==============BASE DATA=========================================
date_range = pd.date_range(start="2022-01-01", end="2025-12-31", freq="D")
trip_dates = rng.choice(date_range, size=N_ROWS)

rows = []
route_list = list(WORLD["route_weights"].keys())
route_weight_list = list(WORLD["route_weights"].values())
route_to_entity = {r: e for e, routes in WORLD["routes_by_entity"].items() for r in routes}
vehicle_weights_by_tier = {tier: WORLD["vehicle_weights_by_tier"][tier] for tier in WORLD["vehicle_weights_by_tier"]}
for i in range(N_ROWS):
    trip_date = pd.Timestamp(trip_dates[i])
    route   = rng.choice(route_list, p=route_weight_list)
    entity  = route_to_entity[route]
    tier    = demand_tier(WORLD["route_multiplier"][route])
    vehicle = rng.choice(WORLD["vehicle_types"], p=vehicle_weights_by_tier[tier])

    # weekday (commuter) ridership runs higher than weekend
    is_weekend = trip_date.dayofweek >= 5
    base_ridership = (
        rng.normal(loc=42, scale=13) if not is_weekend
        else rng.normal(loc=24, scale=11)
    )

    # Demand is what people want; ridership is what the vehicle can carry.
    # Multipliers stack: vehicle size x route demand x season.
    demand = (
        base_ridership
        * WORLD["vehicle_multiplier"][vehicle]
        * WORLD["route_multiplier"][route]
        * WORLD["seasonality_by_quarter"][trip_date.quarter]
    )
    ridership = max(0, min(round(demand), WORLD["vehicle_capacity"][vehicle]))
    rows.append({
        "TripID": f"TRIP-{10000 + i}",
        "TripDate": trip_date,
        WORLD["entity_col"]: entity,
        WORLD["route_col"]: route,
        "VehicleType": vehicle,
        "Ridership": ridership,
    })

df = pd.DataFrame(rows)
print(f"Base clean dataset: {len(df)} rows")

# ==============ERROR FUNCTIONS=========================================
# STRUCTURAL (adds/removes rows) runs before the shuffle.
# CELL-LEVEL (changes values in existing rows) runs after the shuffle.

# ---Pick Targets to add errors to
def pick_targets(df, n, used, rng, pool=None):
    if pool is None:
        pool = df.index
    pool = pool[~pool.isin(used)]
    idx = rng.choice(pool, size=n, replace=False)
    used.update(idx.tolist())
    return idx

# --- structural: exact duplicate rows ---
def plant_exact_dupes(df, cfg, rng):
    n = cfg["n"]
    src_idx = rng.choice(df.index, size=n, replace=False)
    trip_ids = df.loc[src_idx, "TripID"].tolist()
    df = pd.concat([df, df.loc[src_idx].copy()], ignore_index=True)
    print(f"Exact duplicate rows: planted {n}")
    return df, trip_ids


# --- structural: conflicting duplicates (same TripID, different Ridership) ---
def plant_conflicting_dupes(df, cfg, protected_ids, rng):
    n = cfg["n"]
    pool = df.index[~df["TripID"].isin(protected_ids)]
    src_idx = rng.choice(pool, size=n, replace=False)
    trip_ids = df.loc[src_idx, "TripID"].tolist()
    dupe_rows = df.loc[src_idx].copy()
    for idx in dupe_rows.index:
        cap = WORLD["vehicle_capacity"][dupe_rows.loc[idx, "VehicleType"]]
        bumped = int(dupe_rows.loc[idx, "Ridership"]) + rng.choice([-9, -6, 5, 7, 10])
        dupe_rows.loc[idx, "Ridership"] = max(0, min(bumped, cap))
    df = pd.concat([df, dupe_rows], ignore_index=True)
    print(f"Conflicting duplicate TripIDs: planted {n} (Remove Duplicates won't catch these)")
    return df, trip_ids


# --- structural: one entity stops reporting after a cutoff date ---
def plant_data_dropout(df, cfg):
    entity = cfg["entity"]
    cutoff = pd.Timestamp("2025-01-01") + pd.Timedelta(days=cfg["cutoff_day_of_year"] - 1)
    mask = (df[WORLD["entity_col"]] == entity) & (df["TripDate"] >= cutoff)
    n_removed = int(mask.sum())
    df = df[~mask].reset_index(drop=True)
    print(f"Data dropout: removed {n_removed} {entity} rows from {cutoff.date()} onward")
    return df


# --- cell-level: inconsistent entity naming ---
def plant_state_naming(df, cfg, used, rng):
    n = cfg["n"]
    variants = WORLD["entity_naming_variants"]
    eligible = df.index[df[WORLD["entity_col"]].isin(variants.keys())]
    idx = pick_targets(df, n, used, rng, pool=eligible)
    for i in idx:
        df.loc[i, WORLD["entity_col"]] = rng.choice(variants[df.loc[i, WORLD["entity_col"]]])
    print(f"Inconsistent {WORLD['entity_col']} naming: planted {n}")
    return df


# --- cell-level: inconsistent route naming ---
def plant_route_naming(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    for i in idx:
        original = df.loc[i, WORLD["route_col"]]
        variant = rng.choice(["trailing_space", "lowercase", "dash_format"])
        if variant == "trailing_space":
            df.loc[i, WORLD["route_col"]] = original + " "
        elif variant == "lowercase":
            df.loc[i, WORLD["route_col"]] = original.lower()
        else:
            df.loc[i, WORLD["route_col"]] = original.replace("Route ", "RT ")
    print(f"Inconsistent {WORLD['route_col']} naming: planted {n}")
    return df

# --- cell-level: Ridership stored as text ---
def plant_text_numbers(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    df["Ridership"] = df["Ridership"].astype(object)  # allow mixed str/num in this column
    for i in idx:
        val = df.loc[i, "Ridership"]
        # "plain" has no visible character difference - it just silently becomes text
        style = rng.choice(["plain", "plain", "leading_space", "tilde"])
        if style == "plain":
            df.loc[i, "Ridership"] = str(val)
        elif style == "leading_space":
            df.loc[i, "Ridership"] = f" {val}"
        else:
            df.loc[i, "Ridership"] = f"~{val}"
    print(f"Numbers stored as text (Ridership): planted {n}")
    return df

# --- cell-level: blank Ridership ---
def plant_blanks(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    df.loc[idx, "Ridership"] = np.nan
    print(f"Blank Ridership: planted {n}")
    return df


# --- cell-level: outlier ridership (fat-finger, 100x too big) ---
def plant_outlier_ridership(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    for i in idx:
        df.loc[i, "Ridership"] = int(df.loc[i, "Ridership"]) * 100
    print(f"Outlier Ridership (100x): planted {n}")
    return df


# --- cell-level: negative ridership ---
def plant_negative_ridership(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    for i in idx:
        val = df.loc[i, "Ridership"]
        val = int(val) if pd.notna(val) and val != 0 else 5
        df.loc[i, "Ridership"] = -abs(val)
    print(f"Negative Ridership: planted {n}")
    return df


# --- cell-level: ridership over vehicle capacity ---
def plant_over_capacity(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    for i in idx:
        cap = WORLD["vehicle_capacity"][df.loc[i, "VehicleType"]]
        df.loc[i, "Ridership"] = cap + rng.integers(12, 36)
    print(f"Ridership over vehicle capacity: planted {n}")
    return df


# --- cell-level: wrong-year dates ---
def plant_wrong_year_dates(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    for i in idx:
        ts = df.loc[i, "TripDate"]
        df.loc[i, "TripDate"] = ts.replace(year=2024)
    print(f"Wrong-year dates: planted {n}")
    return df


# --- cell-level: route logged under the wrong entity ---
def plant_route_state_mismatch(df, cfg, used, rng):
    n = cfg["n"]
    idx = pick_targets(df, n, used, rng)
    for i in idx:
        true_entity = df.loc[i, WORLD["entity_col"]]
        wrong_entity = rng.choice([e for e in WORLD["entities"] if e != true_entity])
        df.loc[i, WORLD["entity_col"]] = wrong_entity
    print(f"{WORLD['route_col']}/{WORLD['entity_col']} mismatch: planted {n}")
    return df

# ==============RUN PIPELINE=========================================
# structural errors -> shuffle -> cell-level errors

protected_ids = []  # TripIDs from structural dupe errors, cell-level errors skip these

if ERROR_CONFIG["exact_dupes"]["include"]:
    df, ids = plant_exact_dupes(df, ERROR_CONFIG["exact_dupes"], rng)
    protected_ids += ids

if ERROR_CONFIG["conflicting_dupes"]["include"]:
    df, ids = plant_conflicting_dupes(df, ERROR_CONFIG["conflicting_dupes"], protected_ids, rng)
    protected_ids += ids

if ERROR_CONFIG["data_dropout"]["include"]:
    df = plant_data_dropout(df, ERROR_CONFIG["data_dropout"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # row order now final

used = set(df.index[df["TripID"].isin(protected_ids)].tolist())

cell_level_steps = [
    ("state_naming", plant_state_naming),
    ("route_naming", plant_route_naming),
    ("text_numbers", plant_text_numbers),
    ("blanks", plant_blanks),
    ("outlier_ridership", plant_outlier_ridership),
    ("negative_ridership", plant_negative_ridership),
    ("over_capacity", plant_over_capacity),
    ("wrong_year_dates", plant_wrong_year_dates),
    ("route_state_mismatch", plant_route_state_mismatch),
]

for key, func in cell_level_steps:
    cfg = ERROR_CONFIG[key]
    if cfg["include"]:
        df = func(df, cfg, used, rng)

print(f"\nFinal dataset: {len(df)} rows")

# ==============SAVE FILE=========================================
df.to_excel(f"/content/{RUN_LABEL}_data.xlsx", index=False, sheet_name="BusTrips")
df.to_csv(f"/content/{RUN_LABEL}_data.csv", index=False)
print(f"Saved: {RUN_LABEL} _data.xlsx & .csv")

Base clean dataset: 25000 rows

Final dataset: 25000 rows
Saved: Bus Depot ML2 _data.xlsx & .csv
